# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 9b — Mask-based explanation, and agreement over training

Two measurements are missing from Part 9. The mask-based explainer failed because the
explanation library calls a model with a different signature from the one used here, and the
curve of agreement over the course of training was recorded for a single run.

Both use the twenty models already saved, so nothing is trained from scratch except the four
additional runs needed to repeat the training curve.

**Contents**

1. Environment and setup
2. Mask-based explanation across settings and models
3. Agreement over training, repeated
4. Conformal coverage for the descriptor model
5. Summary

*Approximate runtime: two to three hours on a T4.*

## 1. Environment and setup

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 26.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00


The shared definitions are written out here as well, so this notebook runs on a fresh
session without any other notebook having been opened first.

In [2]:
%%writefile /content/ames9.py
"""Definitions shared across Part 9, matching the protocol used in Parts 2 to 5."""
import random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator as rfg
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*"); warnings.filterwarnings("ignore")
FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rfg.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return ""
    try: return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try: return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception: return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values()); rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr)+len(g) <= a: tr += g
        elif len(tr)+len(va)+len(g) <= b: va += g
        else: te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None: return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr

def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None: X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)

def one_hot(x, allowable):
    if x not in allowable: x = allowable[-1]
    return [int(x == a) for a in allowable]

def atom_features(a):
    f  = one_hot(a.GetSymbol(), ATOM_LIST) + one_hot(str(a.GetHybridization()), HYBRIDIZATION)
    f += [a.GetDegree(), a.GetFormalCharge(), int(a.GetIsAromatic()),
          a.GetTotalNumHs(), int(a.IsInRing())]
    return f

def bond_features(b):
    bt = b.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]

def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i, j], [j, i]]; ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)

def build_dataset(d):
    g = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [x for x in g if x is not None]

class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn": conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else: raise ValueError(kind)
            self.convs.append(conv); self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(x, ei, ea) if self.use_edge_attr else conv(x, ei)
            h = F.relu(bn(h)); h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, b), global_add_pool(x, b)], dim=1)
        return self.head(g).squeeze(-1)

def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4, use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads, use_edge_attr=use_edge_attr).to(DEVICE)

class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = x, edge_index, edge_attr, batch

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for d in loader:
        d = d.to(DEVICE)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(model(d)).cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True, epoch_hook=None):
    """The protocol used for every model in this study."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"],
                           weight_decay=params["weight_decay"])
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    tl = DataLoader(tr_set, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    vl = DataLoader(va_set, batch_size=256)
    best, best_state, waited = -1.0, None, 0
    for ep in range(max_epochs):
        model.train()
        for d in tl:
            d = d.to(DEVICE); opt.zero_grad()
            crit(model(d), d.y.view(-1)).backward(); opt.step()
        y, p = evaluate(model, vl)
        auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else 0.5
        if epoch_hook is not None: epoch_hook(ep, model, auc)
        if auc > best:
            best, waited = auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience: break
    model.load_state_dict(best_state); model.eval()
    return model, best

def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0, weight_decay=0.0,
                 lr=None):
    """Fit a target with regularisation and early stopping switched off."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout,
                      params.get("residual", True))
    opt = torch.optim.Adam(model.parameters(), lr=lr or params["lr"],
                           weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()
    ld = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True, drop_last=True)
    last = float("nan")
    for ep in range(n_epochs):
        model.train(); tot = 0.0
        for d in ld:
            d = d.to(DEVICE); opt.zero_grad()
            loss = crit(model(d), d.y.view(-1)); loss.backward(); opt.step()
            tot += loss.item()
        last = tot / max(len(ld), 1)
    model.eval(); return model, last

Writing /content/ames9.py


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL, MODELS = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain", f"{WORK}/models"
CKPT = f"{WORK}/ckpt_part9"
for d in (R, EXPL, CKPT): os.makedirs(d, exist_ok=True)

import sys, json, time, glob
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames9 import *
from sklearn.metrics import roc_auc_score

SEED, PRIMARY_SPLIT_SEED = 42, 101
MODEL_SEEDS = list(range(20000, 20020))
K_FRAC, N_IG_STEPS, N_TIEBREAK = 0.30, 32, 200
HP_GNN = json.load(open(f"{R}/hpo_gnn.json")); hp = HP_GNN["gin"]
print("device", DEVICE)

Mounted at /content/drive
device cuda


In [4]:
df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
assert set(df_test["canonical"]) == set(pd.read_csv(f"{R}/primary_test.csv")["canonical"])
X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values
train_set, valid_set = build_dataset(df_train), build_dataset(df_valid)
POS_W = torch.tensor([(y_train == 0).sum()/max((y_train == 1).sum(),1)],
                     dtype=torch.float, device=DEVICE)

TOXICOPHORES = {
    "aromatic_nitro":"[a][N+](=O)[O-]", "aromatic_amine":"[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":"[a][NX2]=[OX1]", "alkyl_nitrite":"[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":"[NX3][NX2]=[OX1]", "epoxide":"C1OC1", "aziridine":"C1NC1",
    "azide":"[NX2]=[NX2+]=[NX1-]", "azo_aromatic":"[a][NX2]=[NX2][a]",
    "aliphatic_halide":"[CX4][F,Cl,Br,I]", "hetero_hetero_bond":"[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":"c1ccc2ccccc2c1", "aromatic_hydroxylamine":"[a][NX3;H1][OX2;H1]",
    "michael_acceptor":"[CX3]=[CX3][CX3]=[OX1]", "aromatic_azoxy":"[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":"[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":"[a][NX3;H1][NX3;H2]", "alkyl_sulfonate":"[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
EVAL = []
for smi, lab in zip(df_test["canonical"], y_test):
    if lab != 1: continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0: continue
    tox = set()
    for p in PATTERNS.values():
        for m in mol.GetSubstructMatches(p): tox.update(m)
    if not tox or len(tox) == mol.GetNumAtoms(): continue
    EVAL.append({"smiles": smi, "mol": mol, "tox": tox, "n_atoms": mol.GetNumAtoms()})
GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
CHANCE = np.array([len(e["tox"])/e["n_atoms"] for e in EVAL]); N_EVAL = len(EVAL)

def overlap(scores, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed); out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores)):
        if sc is None or len(sc) != e["n_atoms"]: continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac*n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"])/k
    return out

def lift(ov):
    m = ~np.isnan(ov)
    return float(ov[m].mean()/CHANCE[m].mean()) if m.sum() else np.nan

def scored(scores, k_frac=K_FRAC, n=N_TIEBREAK, base=0):
    return np.array([lift(overlap(scores, k_frac, base+j)) for j in range(n)])

DEG = [np.array([a.GetDegree() for a in e["mol"].GetAtoms()], dtype=float) for e in EVAL]
DEG_LIFT = scored(DEG, base=4000).mean()
print(f"evaluation set {N_EVAL}; degree baseline {DEG_LIFT:.4f}")

def grad_scores(model, graphs, method="ig", n_steps=N_IG_STEPS, batch_size=64):
    """Attribution scores; the model's gradient state is saved and restored so this can be
    called on a model that is still being trained."""
    was_training = model.training
    saved = [p.requires_grad for p in model.parameters()]
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE); x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for al in torch.linspace(1.0/n_steps, 1.0, n_steps):
                xa = (x0*al).detach().requires_grad_(True)
                lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(lg.sum(), xa)[0]
            attr = x0 * (total/n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            lg = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            g = torch.autograd.grad(lg.sum(), xa)[0]
            attr = g if method == "saliency" else x0*g
        sc = attr.abs().sum(dim=1).detach().cpu().numpy(); b = batch.batch.cpu().numpy()
        for gi in range(batch.num_graphs): out.append(sc[b == gi])
    for p, f in zip(model.parameters(), saved): p.requires_grad_(f)
    model.train(was_training)
    return out

def load_model(sd):
    m = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"], hp.get("residual", True))
    m.load_state_dict(torch.load(f"{MODELS}/gin_attr_seed{sd}.pt",
                                 map_location=DEVICE, weights_only=False)["state_dict"])
    m.eval(); return m

evaluation set 237; degree baseline 1.2136


## 2. Mask-based explanation across settings and models

The explanation library calls a model as `model(x, edge_index, ...)`, while the networks here
take a single data object. A thin adapter reconciles the two, which is what the earlier
attempt was missing.

Three settings are run on five models. The entropy of the learned mask is recorded alongside
the agreement, because a mask that stays close to uniform induces an arbitrary ranking rather
than an uninformative one, and the two would otherwise be indistinguishable in the score.

In [5]:
import torch.nn as nn
from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.data import Batch

class ExplainAdapter(nn.Module):
    """Presents the signature the explanation library expects."""
    def __init__(self, model):
        super().__init__(); self.model = model
    def forward(self, x, edge_index, edge_attr=None, batch=None, **kw):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        out = self.model(TensorView(x, edge_index, edge_attr, batch))
        return out.view(-1, 1)

def run_masks(model, graphs, cfg):
    alg = GNNExplainer(epochs=cfg["epochs"], lr=cfg["lr"])
    try:
        alg.coeffs.update({"node_feat_size": cfg["node_feat_size"],
                           "node_feat_ent":  cfg["node_feat_ent"]})
    except Exception:
        pass
    ex = Explainer(model=ExplainAdapter(model).to(DEVICE), algorithm=alg,
                   explanation_type="model", node_mask_type="attributes",
                   edge_mask_type="object",
                   model_config=dict(mode="binary_classification",
                                     task_level="graph", return_type="raw"))
    scores, ent = [], []
    for g in graphs:
        b = Batch.from_data_list([g]).to(DEVICE)
        e = ex(b.x, b.edge_index, edge_attr=b.edge_attr, batch=b.batch)
        nm = e.node_mask.abs().sum(dim=1).detach().cpu().numpy()
        scores.append(nm)
        s = nm.sum()
        p = nm/s if s > 0 else np.full(len(nm), 1.0/len(nm))
        p = np.clip(p, 1e-12, 1.0)
        ent.append(float(-(p*np.log(p)).sum()/np.log(len(nm))) if len(nm) > 1 else np.nan)
    return scores, np.array(ent)

# one molecule first, so a signature problem surfaces in seconds rather than hours
probe = load_model(MODEL_SEEDS[0])
sc, en = run_masks(probe, GRAPHS[:1], {"epochs":20,"lr":0.01,"node_feat_size":1.0,"node_feat_ent":0.1})
print(f"probe succeeded: {len(sc[0])} atom scores, entropy {en[0]:.3f}")
del probe
if torch.cuda.is_available(): torch.cuda.empty_cache()

probe succeeded: 19 atom scores, entropy 0.993


In [6]:
SETTINGS = [{"name":"default","epochs":100,"lr":0.01,"node_feat_size":1.0,"node_feat_ent":0.1},
            {"name":"longer","epochs":300,"lr":0.01,"node_feat_size":1.0,"node_feat_ent":0.1},
            {"name":"weak_reg","epochs":100,"lr":0.01,"node_feat_size":0.1,"node_feat_ent":0.01}]
GX_PATH = f"{R}/table48_gnnexplainer_settings.csv"
rows = (pd.read_csv(GX_PATH).to_dict("records")
        if os.path.exists(GX_PATH) and os.path.getsize(GX_PATH) > 10 else [])
done = {(int(r["seed"]), r["setting"]) for r in rows}
t0 = time.time()
for sd in MODEL_SEEDS[:5]:
    todo = [c for c in SETTINGS if (sd, c["name"]) not in done]
    if not todo: continue
    m = load_model(sd)
    for cfg in todo:
        s0 = time.time()
        sc, ent = run_masks(m, GRAPHS, cfg)
        np.savez_compressed(f"{EXPL}/masks_{cfg['name']}_seed{sd}.npz",
                            **{f"m{i}": x for i, x in enumerate(sc)})
        rows.append({"seed": sd, "setting": cfg["name"], "epochs": cfg["epochs"],
                     "node_feat_size": cfg["node_feat_size"],
                     "node_feat_ent": cfg["node_feat_ent"],
                     "lift": scored(sc, n=50, base=90_000+sd).mean(),
                     "mask_entropy_mean": float(np.nanmean(ent)),
                     "mask_entropy_sd": float(np.nanstd(ent)),
                     "minutes": (time.time()-s0)/60})
        pd.DataFrame(rows).to_csv(GX_PATH, index=False)
        print(f"  seed {sd} {cfg['name']:9s} lift {rows[-1]['lift']:.4f}  "
              f"entropy {rows[-1]['mask_entropy_mean']:.3f}  "
              f"({rows[-1]['minutes']:.1f} min, total {(time.time()-t0)/60:.0f})")
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()

gx = pd.DataFrame(rows)
print()
print(gx.groupby("setting")[["lift","mask_entropy_mean"]].agg(["mean","std","count"]).round(4).to_string())
print(f"\n  degree baseline {DEG_LIFT:.4f}; a lift of one is what a random ranking gives.")
print("  a mask entropy near 1.0 means the mask is close to uniform, so the ranking it")
print("  induces is arbitrary rather than informative.")

  seed 20000 default   lift 0.8968  entropy 0.995  (4.3 min, total 4)
  seed 20000 longer    lift 1.0188  entropy 0.971  (12.7 min, total 17)
  seed 20000 weak_reg  lift 0.8554  entropy 0.995  (4.3 min, total 21)
  seed 20001 default   lift 0.9188  entropy 0.996  (4.4 min, total 26)
  seed 20001 longer    lift 1.1161  entropy 0.992  (12.8 min, total 39)
  seed 20001 weak_reg  lift 0.9035  entropy 0.995  (4.4 min, total 43)
  seed 20002 default   lift 1.1219  entropy 0.997  (4.4 min, total 47)
  seed 20002 longer    lift 1.1862  entropy 0.983  (12.8 min, total 60)
  seed 20002 weak_reg  lift 0.9793  entropy 0.996  (4.4 min, total 64)
  seed 20003 default   lift 1.0218  entropy 0.994  (4.4 min, total 69)
  seed 20003 longer    lift 1.0224  entropy 0.971  (12.8 min, total 82)
  seed 20003 weak_reg  lift 0.9443  entropy 0.994  (4.3 min, total 86)
  seed 20004 default   lift 0.8889  entropy 0.994  (4.4 min, total 90)
  seed 20004 longer    lift 0.9562  entropy 0.978  (12.7 min, total 103)
 

## 3. Agreement over training, repeated

Agreement with the alerts was recorded once over the course of a single run and rose sharply
in the first epochs before falling away. Four further runs establish whether that shape is a
property of training or of that seed.

In [7]:
EPOCH_PATH = f"{R}/table45b_lift_by_epoch_repeats.csv"
MARKS = [0, 1, 2, 3, 5, 8, 12, 18, 25, 35, 50, 70, 100, 140, 190, 249]
RUN_SEEDS = [31337, 31338, 31339, 31340, 31341]
rows = pd.read_csv(EPOCH_PATH).to_dict("records") if os.path.exists(EPOCH_PATH) else []
done = {int(r["seed"]) for r in rows}
t0 = time.time()
for sd in RUN_SEEDS:
    if sd in done: continue
    rec = []
    def hook(ep, model, auc):
        if ep in MARKS:
            sc = grad_scores(model, GRAPHS, "ig")
            rec.append({"seed": sd, "epoch": ep+1, "valid_auc": auc,
                        "lift": scored(sc, n=50, base=9000+ep).mean()})
    set_seed(sd)
    m0 = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"], hp.get("residual", True))
    rec.append({"seed": sd, "epoch": 0, "valid_auc": np.nan,
                "lift": scored(grad_scores(m0, GRAPHS, "ig"), n=50, base=8999).mean()})
    del m0
    m, _ = train_gnn("gin", hp, sd, train_set, valid_set, POS_W, epoch_hook=hook)
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    rows += rec
    pd.DataFrame(rows).to_csv(EPOCH_PATH, index=False)
    print(f"  seed {sd}: {len(rec)} checkpoints ({(time.time()-t0)/60:.1f} min)")

ep = pd.DataFrame(rows)
piv = ep.pivot_table(index="epoch", columns="seed", values="lift")
summ = ep.groupby("epoch")["lift"].agg(["mean","std","count"]).join(
       ep.groupby("epoch")["valid_auc"].mean().rename("valid_auc"))
summ["degree_baseline"] = DEG_LIFT
summ.to_csv(f"{R}/table45c_lift_by_epoch_summary.csv")
print()
print(summ.round(4).to_string())
peak = summ["mean"].idxmax()
print(f"\n  agreement peaks at epoch {peak} ({summ.loc[peak,'mean']:.4f}) and ends at "
      f"{summ['mean'].iloc[-1]:.4f}, while validation ROC-AUC ends at "
      f"{summ['valid_auc'].iloc[-1]:.4f}")

  seed 31337: 12 checkpoints (2.3 min)
  seed 31338: 11 checkpoints (4.1 min)
  seed 31339: 13 checkpoints (7.1 min)
  seed 31340: 13 checkpoints (10.2 min)
  seed 31341: 12 checkpoints (12.6 min)

         mean     std  count  valid_auc  degree_baseline
epoch                                                   
0      1.1095  0.0568      5        NaN           1.2136
1      1.1239  0.0828      5     0.7333           1.2136
2      1.1719  0.1391      5     0.7578           1.2136
3      1.0778  0.0522      5     0.7808           1.2136
4      1.1194  0.1042      5     0.7875           1.2136
6      1.1650  0.0970      5     0.8133           1.2136
9      1.1769  0.1396      5     0.8134           1.2136
13     1.1689  0.1084      5     0.8188           1.2136
19     1.1814  0.1540      5     0.8188           1.2136
26     1.2019  0.0932      5     0.8253           1.2136
36     1.1950  0.0951      5     0.8317           1.2136
51     1.1810  0.1117      4     0.8279           1.2136
71  

## 4. Conformal coverage for the descriptor model

The descriptor model is absent from the conformal table because its validation predictions
were not retained. They are regenerated here.

In [8]:
from xgboost import XGBClassifier
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from scipy import stats

names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(names)
DP = f"{R}/rdkit2d_descriptors.npz"
if os.path.exists(DP):
    z = np.load(DP, allow_pickle=True); DESC, SM = z["desc"], list(z["smiles"])
else:
    DESC = np.nan_to_num(np.array([list(calc.CalcDescriptors(Chem.MolFromSmiles(s)))
                                   for s in df["canonical"]], dtype=float),
                         nan=0.0, posinf=0.0, neginf=0.0)
    SM = list(df["canonical"]); np.savez(DP, desc=DESC, smiles=np.array(SM, dtype=object))
IDX = {s: i for i, s in enumerate(SM)}
def dfor(frame): return DESC[[IDX[s] for s in frame["canonical"]]]
HP_DESC = json.load(open(f"{R}/hpo_descriptor.json"))
Dtr, Dva, Dte = dfor(df_train), dfor(df_valid), dfor(df_test)

def fit_desc(seed):
    m = XGBClassifier(n_estimators=1500, **HP_DESC,
                      scale_pos_weight=(y_train==0).sum()/max((y_train==1).sum(),1),
                      eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                      random_state=seed, n_jobs=-1)
    m.fit(Dtr, y_train, eval_set=[(Dva, y_valid)], verbose=False); return m
pv = np.mean([fit_desc(s).predict_proba(Dva)[:,1] for s in [42,43,44]], axis=0)
pt = np.mean([fit_desc(s).predict_proba(Dte)[:,1] for s in [42,43,44]], axis=0)
np.savez(f"{R}/predictions_extra.npz",
         **{**dict(np.load(f"{R}/predictions_extra.npz")),
            "p_descriptor_valid": pv, "p_descriptor": pt})

ALPHA = 0.10
def conformal(pv_, yv_, pt_, scheme):
    if scheme == "marginal":
        cal = np.where(yv_ == 1, 1-pv_, pv_); n = len(cal)
        q = np.quantile(cal, min(1.0, np.ceil((n+1)*(1-ALPHA))/n), method="higher")
        return (1-pt_) <= q, pt_ <= q
    cal = {c: np.sort(np.where(c == 1, 1-pv_[yv_ == c], pv_[yv_ == c])) for c in (0,1)}
    inc = {}
    for c in (0,1):
        s = 1-pt_ if c == 1 else pt_; n = len(cal[c])
        inc[c] = np.array([(np.sum(cal[c] >= si)+1)/(n+1) for si in s]) > ALPHA
    return inc[1], inc[0]

rows = []
for scheme in ["marginal","mondrian"]:
    ip, ineg = conformal(pv, y_valid, pt, scheme)
    cov = np.where(y_test == 1, ip, ineg); size = ip.astype(int)+ineg.astype(int)
    k, N = int(cov.sum()), len(cov)
    rows.append({"model":"XGBoost (RDKit-2D)","scheme":scheme,"nominal_coverage":1-ALPHA,
                 "coverage":cov.mean(),
                 "coverage_ci_low": stats.beta.ppf(0.025,k,N-k+1) if k else 0.0,
                 "coverage_ci_high": stats.beta.ppf(0.975,k+1,N-k) if k < N else 1.0,
                 "p_vs_nominal": stats.binomtest(k,N,1-ALPHA).pvalue,
                 "avg_set_size":size.mean(),"singleton_rate":(size==1).mean(),
                 "uncertain_rate":(size==2).mean(),"empty_rate":(size==0).mean()})
cf = pd.DataFrame(rows)
cf.to_csv(f"{R}/table51_descriptor_conformal.csv", index=False)
print(cf.round(4).to_string(index=False))

             model   scheme  nominal_coverage  coverage  coverage_ci_low  coverage_ci_high  p_vs_nominal  avg_set_size  singleton_rate  uncertain_rate  empty_rate
XGBoost (RDKit-2D) marginal               0.9    0.8774           0.8513            0.9004        0.0475        1.3843          0.6157          0.3843         0.0
XGBoost (RDKit-2D) mondrian               0.9    0.8815           0.8558            0.9041        0.1073        1.3664          0.6336          0.3664         0.0


## 5. Summary

In [9]:
print("mask-based explanation")
if len(gx):
    print(gx.groupby("setting")[["lift","mask_entropy_mean"]].mean().round(4).to_string())
print(f"  degree baseline {DEG_LIFT:.4f}")
print("\nagreement over training")
print(summ[["mean","valid_auc"]].round(4).to_string())
print("\nWritten to", R)
for f in ["table48_gnnexplainer_settings.csv","table45b_lift_by_epoch_repeats.csv",
          "table45c_lift_by_epoch_summary.csv","table51_descriptor_conformal.csv"]:
    print("  -", f)
import shutil
from google.colab import files
shutil.make_archive("/content/part9b_results", "zip", R)
files.download("/content/part9b_results.zip")

mask-based explanation
            lift  mask_entropy_mean
setting                            
default   0.9696             0.9952
longer    1.0600             0.9791
weak_reg  0.9112             0.9944
  degree baseline 1.2136

agreement over training
         mean  valid_auc
epoch                   
0      1.1095        NaN
1      1.1239     0.7333
2      1.1719     0.7578
3      1.0778     0.7808
4      1.1194     0.7875
6      1.1650     0.8133
9      1.1769     0.8134
13     1.1689     0.8188
19     1.1814     0.8188
26     1.2019     0.8253
36     1.1950     0.8317
51     1.1810     0.8279
71     1.1885     0.8294

Written to /content/drive/MyDrive/ames_pipeline/results
  - table48_gnnexplainer_settings.csv
  - table45b_lift_by_epoch_repeats.csv
  - table45c_lift_by_epoch_summary.csv
  - table51_descriptor_conformal.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>